In [1]:
from pyspark.sql import SparkSession
import pandas as pd

In [2]:
spark = SparkSession.builder.appName("Case-study-app").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 05:57:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# df1 = pd.read_csv('data/customers.csv')

In [4]:
# df1.head()

In [5]:
# print('Total records:',df1.count())

In [6]:
# df2= pd.read_csv('data/products.csv')
# df3= pd.read_csv('data/order_items.csv')
# df4= pd.read_csv('data/orders.csv')
# df5= pd.read_csv('data/returns.csv')

In [7]:
# print('Total products:', df2.count())

In [8]:
# print('Total orders:', df4.count())

In [9]:
# print('Total returns:', df5.count())

In [10]:
# print('Total order_items:', df3.count())

In [11]:
products= spark.read.option("header",True).csv("data/products.csv")
customers= spark.read.option("header",True).csv("data/customers.csv")
orders= spark.read.option("header",True).csv("data/orders.csv")
results= spark.read.option("header",True).csv("data/returns.csv")
order_items=spark.read.option("header",True).csv("data/order_items.csv")

In [12]:
products.show()

+----------+------------+--------------+-------+---------+
|product_id|product_name|      category|  brand|unit_cost|
+----------+------------+--------------+-------+---------+
|         1|   Product_1|Home & Kitchen|Brand_A|   509.39|
|         2|   Product_2|   Electronics|Brand_A|   332.22|
|         3|   Product_3|        Sports|Brand_D|    68.58|
|         4|   Product_4|      Clothing|Brand_A|   729.19|
|         5|   Product_5|Home & Kitchen|Brand_D|   326.77|
|         6|   Product_6|        Beauty|Brand_E|   684.21|
|         7|   Product_7|          Toys|Brand_D|   634.58|
|         8|   Product_8|Home & Kitchen|Brand_B|   158.47|
|         9|   Product_9|   Electronics|Brand_C|    287.9|
|        10|  Product_10|      Clothing|Brand_A|    93.38|
|        11|  Product_11|         Books|Brand_C|   477.32|
|        12|  Product_12|        Sports|Brand_C|   517.62|
|        13|  Product_13|         Books|Brand_A|   723.64|
|        14|  Product_14|          Toys|Brand_B|   179.4

In [13]:
products.createOrReplaceTempView("Products_table")

In [14]:
result= spark.sql('''
SELECT category, SUM(unit_cost) AS Total_sales
FROM Products_table
GROUP BY category
''')

In [15]:
result.show()

[Stage 6:>                                                          (0 + 1) / 1]

+--------------+------------------+
|      category|       Total_sales|
+--------------+------------------+
|Home & Kitchen| 2901364.330000004|
|        Sports| 2853163.040000003|
|   Electronics|2864604.7399999946|
|      Clothing| 2841424.610000002|
|         Books|2853871.8500000075|
|        Beauty|2919388.7500000037|
|          Toys|2851913.1100000013|
+--------------+------------------+



In [16]:
result.write.mode("overwrite").csv("output/q2",header= True)

In [17]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")

In [21]:
res=spark.sql('''
SELECT c.customer_name, SUM(oi.selling_price*oi.quantity) AS Total_amount
FROM order_items oi join orders o
ON oi.order_id = o.order_id
JOIN customers c
ON o.customer_id = c.customer_id
GROUP BY c.customer_name
ORDER BY Total_amount DESC
LIMIT 10
''')

In [23]:
res.show()

26/06/15 05:59:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 05:59:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 38:=============================>                            (1 + 1) / 2]

+--------------+------------------+
| customer_name|      Total_amount|
+--------------+------------------+
|Customer_93094|181569.68000000005|
|Customer_64560|169060.39999999997|
|Customer_23289|          161573.8|
|Customer_52275|153364.78999999998|
|Customer_61218|         153067.55|
|Customer_52034|         152680.05|
|Customer_40442|151037.32000000004|
|Customer_60528|         148691.95|
|Customer_84830|         148363.84|
|Customer_82593|         148281.04|
+--------------+------------------+



In [24]:
res.write.mode("overwrite").csv("output/q3",header= True)

26/06/15 05:59:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 05:59:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

In [27]:
res4=spark.sql('''
SELECT MONTH(order_date) as Month, SUM(selling_price*quantity) AS Total_amount
from orders join order_items
group by MONTH(order_date)
order by Month
''')

In [ ]:
res4 = (
    orders
    .join(order_items, "order_id")
    .groupBy(month(col("order_date")).alias("Month"))
    .agg(
        sum(col("selling_price") * col("quantity")).alias("Total_amount")
    )
    .orderBy("Month")
)

res4.show()